In [1]:
import sys
import os

sys.path.append(os.path.realpath(os.path.join(os.getcwd(), '..', '..', 'T5000-Base', 'Pylians3', 'library')))
sys.path.append(os.path.abspath('../src'))

from pathlib import Path
from tqdm import tqdm

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt

project_root = Path().resolve().parent
sys.path.append(str(project_root))

from src.utils import load_config
from src.validation import get_tpcf

import MAS_library as MASL
import Pk_library as PKL

/home/bartb/venvs/boids/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
WIDTH_LATEX = 4.9
FONT_COLOR = "0" # or "#0E1D35" for the dark blue as per pp theme

In [3]:
def find_equally_spaced_indices(arr, n):
    "Based on specific conditioning parameter."
    sorted_indices = np.argsort(arr)
    sorted_arr = arr[sorted_indices]
    min_val, max_val = sorted_arr[0], sorted_arr[-1]
    division_points = np.linspace(min_val, max_val, n)
    closest_indices = np.searchsorted(sorted_arr, division_points)
    closest_indices = np.clip(closest_indices, 0, len(arr) - 1)
    original_indices = sorted_indices[closest_indices]
    return original_indices

In [4]:
# In and Out Directories
train_run = "20260413_180334"
infer_run = "20260414_140123"
config_dir = f"/gpfs/home4/bartb/T5000/T5000-OVF/results/run_{train_run}/qualitative_2pcf/{infer_run}/final_config.yaml"
config = load_config(config_dir)
output_dir = os.path.dirname(config_dir)

In [5]:
# Load true and generated samples
gen_samples = torch.load(f"/gpfs/home4/bartb/T5000/T5000-OVF/results/run_{train_run}/qualitative_2pcf/{infer_run}/gen_samples.pth", weights_only=False).squeeze(0)
print(f"Shape of generated samples: {gen_samples.shape}")

true_samples = torch.from_numpy(np.load("/gpfs/home4/bartb/T5000/Data/test_halos.npy")[..., :3])
print(f"Shape of true samples: {true_samples.shape}")

# Load parameters
params = pd.read_csv("/gpfs/home4/bartb/T5000/Data/test_cosmology.csv").values
print(f"Shape of parameters: {params.shape}")

Shape of generated samples: (1, 20, 5000, 3)
Shape of true samples: torch.Size([200, 5000, 3])
Shape of parameters: (200, 5)


In [6]:
### equally spaced based on Omega_m! : conditioning[:, 0]
equally_spaced = 20
params_idx_used = find_equally_spaced_indices(params[:,0], equally_spaced)
params_used = params[params_idx_used]
true_samples_used = true_samples[params_idx_used][ :, :, :3] / 1000 # normalize coordinates
true_samples_used = true_samples_used[::4]
params_used = params_used[::4]#[:, [0, -1]]

print(f"True samples used: {true_samples_used.shape}")
print(f"Parameters used: {params_used.shape}, \n {params_used}")

True samples used: torch.Size([5, 5000, 3])
Parameters used: (5, 5), 
 [[0.1001  0.05047 0.8959  1.0103  0.9203 ]
 [0.1839  0.04711 0.8675  1.0421  0.6845 ]
 [0.2683  0.03749 0.7533  0.9845  0.9205 ]
 [0.3539  0.06139 0.6681  0.8327  0.6243 ]
 [0.4365  0.06281 0.7347  1.0699  0.9509 ]]


In [7]:
n_cosmologies = true_samples_used.shape[0]
n_repeats = config["inference"]["n_repeats"]
print(f"Number of cosmologies: {n_cosmologies}, Number of repeats: {n_repeats}") 

Number of cosmologies: 5, Number of repeats: 20


In [8]:
BoxSize = 1000.0
grid = 128
MAS = 'CIC'
n_halos = 5000
shot_noise = BoxSize**3 / n_halos

pos_real = true_samples_used.numpy()
pos_gen = gen_samples

def compute_pk_from_positions_shot_noise(pos_normalized):
    """
    pos_normalized: shape (5000, 3), coordinates in [0, 1]
    Returns k array and P(k) array (shot noise subtracted)
    """
    pos_phys = (pos_normalized * BoxSize).astype(np.float32)
    delta = np.zeros((grid, grid, grid), dtype=np.float32)
    MASL.MA(pos_phys, delta, BoxSize, MAS=MAS)
    delta /= np.mean(delta, dtype=np.float64)
    delta -= 1.0

    devnull = open(os.devnull, 'w')
    old_stdout = sys.stdout
    sys.stdout = devnull
    
    Pk = PKL.Pk(delta, BoxSize, axis=0, MAS=MAS, threads=4)

    sys.stdout = old_stdout
    devnull.close()
    
    return Pk.k3D, Pk.Pk[:, 0] - shot_noise

def compute_pk_from_positions(pos_normalized):
    pos_phys = (pos_normalized * BoxSize).astype(np.float32)
    delta = np.zeros((grid, grid, grid), dtype=np.float32)
    MASL.MA(pos_phys, delta, BoxSize, MAS=MAS)
    delta /= np.mean(delta, dtype=np.float64)
    delta -= 1.0

    devnull = open(os.devnull, 'w')
    old_stdout = sys.stdout
    sys.stdout = devnull
    
    Pk = PKL.Pk(delta, BoxSize, axis=0, MAS=MAS, threads=4)

    sys.stdout = old_stdout
    devnull.close()
    
    return Pk.k3D, Pk.Pk[:, 0]  # no subtraction

pk_gen = {}
pk_true = {}
k_vals = None  # same for all since same grid and box

for idx in range(n_cosmologies):
    # Real
    k_vals, pk_real = compute_pk_from_positions(pos_real[idx])
    pk_true[idx] = pk_real

    # Generated — average over 20 repeats
    pk_gen[idx] = []
    for repeat in range(pos_gen.shape[1]):
        _, pk = compute_pk_from_positions(pos_gen[idx, repeat])
        pk_gen[idx].append(pk)
    
    pk_gen[idx] = np.array(pk_gen[idx])

# -------------------------------------------------------
# Plot — same style as your 2PCF plot
# -------------------------------------------------------

kmax = 0.3  # only trust below this
mask = k_vals < kmax

colors = ['#d62728', '#1f77b4', '#2ca02c', '#9467bd', '#ff7f0e']
fig, ax = plt.subplots(figsize=(WIDTH_LATEX, 2.5))
for i in range(n_cosmologies):
    gen_mean = np.mean(pk_gen[i], axis=0)
    gen_std = np.std(pk_gen[i], axis=0)

    ax.fill_between(k_vals, gen_mean - gen_std, gen_mean + gen_std,
                     alpha=0.25, color=colors[i])
    ax.plot(k_vals, gen_mean, linestyle='--', color=colors[i])
    ax.plot(k_vals, pk_true[i],
            label=rf'$\Omega_m = {params_used[i][0]:.2f} \, \, \, \, \sigma_8 = {params_used[i][-1]:.2f}$',
            color=colors[i])

ax.axhline(shot_noise, color='gray', ls=':', alpha=0.5, label='Shot noise')
ax.set_title("Power Spectrum FlowEGNN", fontsize=8, fontweight="bold")
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r"$k\,[h\,{\rm Mpc}^{-1}]$", color=FONT_COLOR, fontsize=8)
ax.set_ylabel(r'$P(k)$ $[(h^{-1}\,{\rm Mpc})^3]$', color=FONT_COLOR, fontsize=8)
ax.tick_params(colors=FONT_COLOR, which='both', labelsize=6)
ax.legend(labelcolor=FONT_COLOR, fontsize=6, loc='lower left', framealpha=0)
plt.show()

IndexError: index 1 is out of bounds for axis 0 with size 1